# SO-app v0.1 en Kaggle — todo en una celda

Ubuntu 22.04 nativo con `proot` + Chrome real + OpenCode web + túnel Cloudflare.
Corre la única celda de código y espera el `TUNEL: ...` al final.
Primera vez tarda 10-15 min (~300MB de descargas). Re-correr reusa todo.
Requiere Internet activado en el notebook.

In [ ]:
%%bash
set -u
DIR=/kaggle/working/vmdata/proot
mkdir -p "$DIR"
R() { proot -R "$DIR/rootfs" -b /dev -b /proc -b /sys -w / "$@"; }

echo "=== [1/5] proot ==="
command -v proot >/dev/null || { apt-get update -qq && apt-get install -y -qq proot; }

echo "=== [2/5] rootfs ==="
if [ ! -d "$DIR/rootfs/bin" ]; then
  curl -sSL --retry 3 -o "$DIR/ubuntu-base.tar.gz" \
    https://cdimage.ubuntu.com/ubuntu-base/releases/22.04/release/ubuntu-base-22.04-base-amd64.tar.gz
  mkdir -p "$DIR/rootfs"
  tar -xzf "$DIR/ubuntu-base.tar.gz" -C "$DIR/rootfs"
else echo "rootfs existente, reuso"; fi
printf 'nameserver 8.8.8.8\nnameserver 1.1.1.1\n' > "$DIR/rootfs/etc/resolv.conf"
R /bin/sh -c 'apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true update -qq && apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true install -y -qq curl ca-certificates' | tail -1

echo "=== [3/5] apps ==="
if [ ! -x "$DIR/rootfs/opt/google/chrome/chrome" ]; then
  R /bin/sh -c 'curl -sSL --retry 2 -o /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb'
  R /bin/sh -c 'apt-get -o APT::Sandbox::User=root -o Acquire::ForceIPv4=true install -y /tmp/chrome.deb || true; rm -f /tmp/chrome.deb; apt-get -o APT::Sandbox::User=root clean; rm -rf /var/lib/apt/lists/*' | tail -1
else echo "Chrome existente, reuso"; fi
R /bin/sh -c 'LD_LIBRARY_PATH=/usr/lib/x86_64-linux-gnu /opt/google/chrome/chrome --version --no-sandbox'

echo "=== [4/5] opencode web ==="
OCBIN=""; for c in /tools/node/bin/opencode "$(command -v opencode)"; do [ -x "$c" ] && OCBIN="$c" && break; done
[ -z "$OCBIN" ] && { npm install -g opencode-ai 2>&1 | tail -1; OCBIN="$(command -v opencode)"; }
pkill -f "opencode web" 2>/dev/null || true; sleep 1
cd /kaggle/working && nohup "$OCBIN" web --hostname 127.0.0.1 --port 4096 > /tmp/opencode-web.log 2>&1 &
for i in $(seq 1 30); do (echo > /dev/tcp/127.0.0.1/4096) 2>/dev/null && break; sleep 2; done
echo "opencode web en puerto 4096"

echo "=== [5/5] tunel ==="
[ -x /tmp/cloudflared ] || curl -sSL --retry 2 -o /tmp/cloudflared \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  && chmod +x /tmp/cloudflared
pkill -f "cloudflared tunnel" 2>/dev/null || true; sleep 1
nohup /tmp/cloudflared tunnel --url http://127.0.0.1:4096 > /tmp/tunnel.log 2>&1 &
URL=""
for i in $(seq 1 45); do
  URL=$(grep -oE "https://[a-zA-Z0-9.-]+\\.trycloudflare\\.com" /tmp/tunnel.log 2>/dev/null | head -1)
  [ -n "$URL" ] && break; sleep 2
done
echo "TUNEL: $URL"